# Przygotowanie i czyszczenie danych (Data Cleaning)

Celem tego notebooka jest przygotowanie danych sprzedażowych do dalszej analizy.

Dataset pochodzi z publicznego zbioru danych e-commerce (Olist) i zawiera informacje o:

- zamówieniach
- produktach
- klientach
- płatnościach
- sprzedawcach

Na tym etapie wykonujemy:

- wczytanie danych
- wstępną eksplorację
- analizę brakujących wartości
- czyszczenie danych
- połączenie tabel w jeden zbiór analityczny

Efektem końcowym będzie przygotowanie datasetu, który zostanie wykorzystany w dalszej analizie eksploracyjnej (EDA).

## Import bibliotek

Na początku importujemy biblioteki wykorzystywane do analizy danych.

W projekcie korzystamy głównie z:

- pandas — manipulacja danymi
- numpy — operacje numeryczne

In [ ]:
import pandas as pd
import numpy as np

## Wczytanie danych

Dataset Olist składa się z kilku tabel. Najważniejsze z nich to:

- **orders** — informacje o zamówieniach
- **order_items** — produkty w zamówieniach
- **products** — informacje o produktach
- **customers** — informacje o klientach
- **payments** — informacje o płatnościach

Każda z tych tabel zostanie wczytana osobno.

In [ ]:
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

## Wstępna eksploracja danych

Sprawdzamy podstawowe informacje o datasetach:

- liczbę rekordów
- typy kolumn
- przykładowe dane

In [ ]:
orders.head()

In [ ]:
orders.info()

In [ ]:
order_items.head()
products.head()
customers.head()
payments.head()

## Wstępna eksploracja danych

Na tym etapie sprawdzamy podstawowe informacje o danych:

- liczbę rekordów
- typy kolumn
- przykładowe wartości
- strukturę tabel

Pozwala to lepiej zrozumieć dataset przed dalszą analizą.

In [ ]:
def inspect_df(df, name):
    print("DATASET:", name)
    print("------------------")
    print(df.shape)
    print(df.isnull().sum())
    print(df.dtypes)
    print(df.head())

In [ ]:
inspect_df(orders, "orders")
inspect_df(order_items, "order_items")
inspect_df(products, "products")
inspect_df(customers, "customers")
inspect_df(payments, "payments")

## Konwersja dat

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

## Łączenie tabel

Dataset e-commerce jest podzielony na kilka tabel.

Aby przeprowadzić analizę sprzedaży, konieczne jest ich połączenie w jeden zbiór danych.

W tym celu wykorzystujemy operacje **merge** w bibliotece pandas.

Połączone zostaną informacje o:

- zamówieniach
- produktach
- klientach
- płatnościach

In [ ]:
df = order_items.merge(
    orders,
    on="order_id",
    how="left"
)

In [ ]:
df = df.merge(
    products,
    on="product_id",
    how="left"
)

In [ ]:
df = df.merge(
    customers,
    on="customer_id",
    how="left"
)

In [ ]:
df = df.merge(
    payments,
    on="order_id",
    how="left"
)

## Dodanie kolumny revenue

In [ ]:
df["revenue"] = df["price"] + df["freight_value"]

## Sprawdzenie duplikatów

In [ ]:
duplicates = df.duplicated().sum()
print("Number of duplicated rows:", duplicates)

## Analiza brakujących wartości

Sprawdzamy liczbę brakujących wartości w każdej kolumnie.

Brakujące dane są częstym problemem w rzeczywistych datasetach i mogą wpływać na wyniki analizy.

Na podstawie tej analizy zdecydujemy:

- które wartości można uzupełnić
- które rekordy należy usunąć
- które brakujące dane są naturalne i mogą pozostać

In [ ]:
df.isnull().sum()

### Interpretacja brakujących danych

W datasetcie występują brakujące wartości w kilku kolumnach, m.in.:

- informacje o kategorii produktu
- wymiary produktów
- dane dotyczące płatności

W dalszym kroku dane te zostaną odpowiednio przetworzone.

## Czyszczenie danych

Na podstawie wcześniejszej analizy brakujących danych wykonujemy:

- uzupełnienie brakujących kategorii produktów
- usunięcie rekordów z brakującymi informacjami o płatności

In [ ]:
df["product_category_name"] = df["product_category_name"].fillna("unknown")

In [ ]:
df = df.dropna(subset=[
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
])

In [ ]:
df = df.dropna(subset=["payment_value"])

### Analiza brakujących wartości

Zbiór danych zawiera brakujące wartości w kilku kolumnach:

- informacje o kategorii produktu (~1,7 tys. rekordów)
- wymiary produktu (20 rekordów)
- daty dostawy niektórych zamówień
- informacje o płatnościach (3 rekordy)

Strategia postępowania:

- brakujące kategorie produktów zastąpiono wartością „unknown”
- niekompletne rekordy płatności usunięto
- daty dostawy pozostawiono bez zmian, ponieważ odzwierciedlają one rzeczywisty stan zamówień


In [ ]:
df["order_month"] = df["order_purchase_timestamp"].dt.to_period("M")

df["order_year"] = df["order_purchase_timestamp"].dt.year

In [ ]:
df["order_value"] = df.groupby("order_id")["revenue"].transform("sum")

## Zapis przetworzonych danych

Oczyszczony dataset zapisujemy w folderze `data/processed`, aby mógł zostać wykorzystany w kolejnych etapach analizy.

In [ ]:
df.to_csv("../data/processed/sales_cleaned.csv", index=False)